# Day 3: Baseline ML — Vietnamese Price Prediction (<= 1M VND)

**Dataset:** `SeanSunny/items_tv_v6` filtered to price <= 1,000,000 VND

**Metrics:** RMSLE (primary), MAE, MAPE, R2

**Models:** Random, Mean, Median, LR (Arch A + B), RF, XGBoost, LightGBM, CatBoost

In [1]:
!nvidia-smi

Tue Apr 14 17:19:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   40C    P8              9W /  180W |     454MiB /  16311MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#!uv pip uninstall torch torchvision torchaudio

Using Python 3.12.13 environment at: C:\hieu\DACN3\.venv
Uninstalled 3 packages in 1.55s
 - torch==2.9.0
 - torchaudio==2.11.0.dev20260413+cu130
 - torchvision==0.27.0.dev20260413+cu130


In [4]:
#!uv pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130

Using Python 3.12.13 environment at: C:\hieu\DACN3\.venv
Resolved 14 packages in 4.10s
Installed 3 packages in 2.95s
 + torch==2.12.0.dev20260413+cu130
 + torchaudio==2.11.0.dev20260413+cu130
 + torchvision==0.27.0.dev20260413+cu130


In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [2]:
import random
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))

from pricer_vi.items import Item
from pricer_vi.evaluator import evaluate

SEED = 42
DATASET = "SeanSunny/items_tv_v6"
PRICE_THRESHOLD = 1_000_000
CACHE_DIR = Path("day3")
USE_GPU = True  # RTX 5060 Ti

random.seed(SEED)
np.random.seed(SEED)

## 1. Load Data + Filter <= 1M VND

In [3]:
train, val, test = Item.from_hub(DATASET)
print(f"Raw: {len(train):,} train | {len(val):,} val | {len(test):,} test")

train = [item for item in train if item.price <= PRICE_THRESHOLD]
val = [item for item in val if item.price <= PRICE_THRESHOLD]
test = [item for item in test if item.price <= PRICE_THRESHOLD]
print(f"Filtered <= {PRICE_THRESHOLD:,} VND:")
print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"  Sample: {test[0]}")

Raw: 110,000 train | 5,000 val | 5,000 test
Filtered <= 1,000,000 VND:
  Train: 85,727 | Val: 3,926 | Test: 3,872
  Sample: title='Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L HULKER NEW màu Xanh quân đội| Index Living Mall' category='Nhà Cửa - Đời Sống' price=479400 full=None brand=None summary='Tiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.' prompt='Sản phẩm này giá bao nhiêu?\n\nTiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống 

In [4]:
train_prices = [item.price for item in train]
prices = np.array(train_prices, dtype=float)
documents = [item.summary for item in train]

print(f"Price range: {prices.min():,.0f} - {prices.max():,.0f} VND")
print(f"Mean: {prices.mean():,.0f} | Median: {np.median(prices):,.0f} | Std: {prices.std():,.0f}")

# So sanh voi full dataset
print(f"So sanh: Full dataset Mean=1,302,725 | Median=302,000 | Std=3,583,425")
print(f"         1M filter   Mean={prices.mean():,.0f} | Median={np.median(prices):,.0f} | Std={prices.std():,.0f}")

Price range: 4,900 - 1,000,000 VND
Mean: 301,687 | Median: 229,000 | Std: 228,179
So sanh: Full dataset Mean=1,302,725 | Median=302,000 | Std=3,583,425
         1M filter   Mean=301,687 | Median=229,000 | Std=228,179


---
## Step 3: Baselines (Random, Mean, Median)

In [9]:
min_price, max_price = min(train_prices), max(train_prices)

def random_pricer(item):
    return random.randint(min_price, max_price)

random.seed(SEED)
results_random = evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

195,987 22,639 5,125 422,472 167,289 423,313 45,047 189,784 727,146 35,627 445,470 536,546 413,418 524,758 227,939 435,076 317,317 110,774 73,856 119,854 199,342 701,138 311,803 471,162 67,276 294,408 17,396 640,700 128,647 621,292 405,312 365,798 168,048 286,929 546,949 207,604 713,649 747,427 882,286 34,433 554,862 51,186 286,953 310,043 271,679 483,731 117,932 65,672 719,481 208,844 48,825 262,849 315,282 872,686 306,317 674,562 280,995 433,702 42,730 701,235 774,539 550,079 373,641 482,175 64,211 746,996 142,822 428,938 3,756 237,319 639,593 545,076 464,464 483,363 719,465 265,101 60,297 7,529 394,697 223,167 192,050 8,284 6,132 125,520 259,345 319,133 752,765 149,998 804,473 11,807 145,509 97,376 327,335 218,537 780,528 198,454 199,545 406,938 298,428 84,584 497,629 105,846 513,311 688,053 158,651 462,414 170,230 605,120 480,722 35,649 222,286 558,444 737,398 6,239 34,614 342,787 91,040 126,242 571,988 611,490 147,904 93,183 493,770 248,935 312,694 539,365 529,594 35,445 608,926 6

In [10]:
training_average = float(prices.mean())
print(f"Training average: {training_average:,.0f} VND")

def mean_pricer(item):
    return training_average

results_mean = evaluate(mean_pricer, test)

Training average: 301,687 VND


  0%|          | 0/200 [00:00<?, ?it/s]

177,713 202,687 275,687 58,313 175,687 383,313 17,687 39,313 251,687 153,687 32,687 56,687 225,313 249,687 22,313 112,687 171,687 152,687 191,687 78,687 131,813 648,313 78,687 136,687 201,687 2,687 105,687 186,687 513,313 182,687 130,687 222,687 233,687 112,687 225,847 212,687 161,687 132,687 592,313 533,313 6,687 78,187 148,312 163,687 211,688 478,313 251,687 136,687 215,687 152,687 238,437 63,313 213,687 677,313 226,687 82,687 344,871 97,437 23,313 151,687 523,313 81,687 188,687 216,687 101,687 76,687 42,687 214,778 278,313 226,687 66,687 182,687 112,687 148,313 112,687 182,687 248,313 102,687 47,313 687 56,687 388,313 51,687 388,313 252,687 106,013 152,687 202,687 192,687 202,687 247,313 102,687 148,687 588,313 202,687 112,687 73,313 498,313 222,687 161,687 91,687 122,687 74,187 2,687 261,313 79,687 51,687 263,187 111,687 81,687 41,013 90,687 697,313 131,687 153,313 241,687 77,313 547,313 197,687 186,687 139,313 159,687 72,687 205,687 274,313 30,517 12,687 202,687 552,313 131,687 12

In [11]:
training_median = float(np.median(prices))
print(f"Training median: {training_median:,.0f} VND")

def median_pricer(item):
    return training_median

results_median = evaluate(median_pricer, test)

Training median: 229,000 VND


  0%|          | 0/200 [00:00<?, ?it/s]

250,400 130,000 203,000 131,000 103,000 456,000 55,000 112,000 179,000 81,000 40,000 16,000 298,000 177,000 95,000 40,000 99,000 80,000 119,000 6,000 204,500 721,000 6,000 64,000 129,000 70,000 33,000 114,000 586,000 110,000 58,000 150,000 161,000 40,000 153,160 140,000 89,000 60,000 665,000 606,000 66,000 5,500 220,999 91,000 139,001 551,000 179,000 64,000 143,000 80,000 165,750 136,000 141,000 750,000 154,000 10,000 417,558 24,750 96,000 79,000 596,000 9,000 116,000 144,000 29,000 4,000 30,000 287,465 351,000 154,000 6,000 110,000 40,000 221,000 40,000 110,000 321,000 30,000 120,000 72,000 16,000 461,000 21,000 461,000 180,000 178,700 80,000 130,000 120,000 130,000 320,000 30,000 76,000 661,000 130,000 40,000 146,000 571,000 150,000 89,000 19,000 50,000 1,500 70,000 334,000 7,000 21,000 190,500 39,000 9,000 113,700 18,000 770,000 59,000 226,000 169,000 150,000 620,000 125,000 114,000 212,000 87,000 0 133,000 347,000 42,170 60,000 130,000 625,000 59,000 201,000 181,000 270,000 174,000

---
## Step 4: LR + TF-IDF — Architecture A (bigram, no word segmentation)

In [5]:
t0 = time.time()
vectorizer_a = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_a = vectorizer_a.fit_transform(documents)
print(f"TF-IDF A: {X_train_a.shape[0]:,} x {X_train_a.shape[1]:,} ({time.time()-t0:.1f}s)")

selected_words_a = vectorizer_a.get_feature_names_out()
print(f"Sample features: {list(selected_words_a[5000:5020])}")

TF-IDF A: 85,727 x 10,000 (7.4s)
Sample features: ['mạng', 'mạng lan', 'mạng thương', 'mạnh', 'mạnh mẽ', 'mạnh thông', 'mạnh và', 'mảng', 'mảng bám', 'mảnh', 'mảnh ghép', 'mất', 'mầm', 'mẫu', 'mẫu acer', 'mẫu asus', 'mẫu dell', 'mẫu hp', 'mẫu laptop', 'mẫu máy']


In [13]:
t0 = time.time()
lr_model_a = LinearRegression()
lr_model_a.fit(X_train_a, prices)
print(f"LR A train: {time.time()-t0:.1f}s")

def lr_tfidf_arch_a(item):
    x = vectorizer_a.transform([item.summary])
    return max(lr_model_a.predict(x)[0], 0)

results_lr_a = evaluate(lr_tfidf_arch_a, test)

LR A train: 11.4s


  0%|          | 0/200 [00:00<?, ?it/s]

32,369 134,046 216,636 217,232 178,228 459,101 52,114 41,656 272,816 138,262 13,437 19,676 37,438 164,667 66,525 73,106 101,844 287,950 230,887 129,481 54,169 4,288 74,810 79,520 165,267 53,667 196,000 65,291 527,252 67,712 47,615 203,874 96,461 14,785 51,300 3,829 54,045 111,116 323,464 195,074 69,078 156,116 119,765 86,373 214,209 431,833 15,090 80,864 104,223 56,076 211,160 66,930 71,292 419,116 106,339 11,237 11,965 92,264 72,986 263,588 238,429 65,872 38,366 120,851 7,764 45,492 54,359 138,980 105,683 153,296 27,314 15,905 21,872 122,325 47,089 37,956 98,210 11,655 67,705 166,815 51,476 479,891 127,975 127,768 64,512 175,553 72,351 91,718 152,344 17,781 41,615 2,441 133,978 399,827 238,104 21,440 64,808 314,855 197,124 144,445 87,277 66,680 55,520 232,018 86,083 142,855 67,440 38,500 18,146 9,551 32,164 72,431 309,654 313,056 14,501 60,000 51,117 424,932 68,293 9,647 271,021 54,691 60,451 43,745 57,400 72,447 41,205 33,294 357,123 74,312 188,494 109,507 183,308 41,365 29,003 133,3

---
## Step 5: Underthesea Tokenize (1 lan, luu cache .pkl)

Tokenize **train + test** bang underthesea, luu vao  de lan sau khong can tokenize lai.

In [ ]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

CACHE_DIR = Path(".")

CACHE_DIR.mkdir(exist_ok=True)
cache_train = CACHE_DIR / "tokenized_train_1m.pkl"
cache_test = CACHE_DIR / "tokenized_test_1m.pkl"

# --- Tokenize train ---
if cache_train.exists():
    print(f"Loading cached: {cache_train}")
    with open(cache_train, "rb") as f:
        tokenized_train = pickle.load(f)
    print(f"  Loaded {len(tokenized_train):,} docs")
else:
    t0 = time.time()
    print(f"Tokenizing {len(documents):,} train docs (4 workers)...")
    with Pool(4) as p:
        tokenized_train = list(tqdm(
            p.imap(tokenize_one, documents, chunksize=500),
            total=len(documents), desc="Tokenize train",
        ))
    print(f"  Done: {time.time()-t0:.1f}s")
    with open(cache_train, "wb") as f:
        pickle.dump(tokenized_train, f)
    print(f"  Saved: {cache_train}")

# --- Tokenize test ---
test_summaries = [item.summary for item in test]
if cache_test.exists():
    print(f"Loading cached: {cache_test}")
    with open(cache_test, "rb") as f:
        tokenized_test = pickle.load(f)
    print(f"  Loaded {len(tokenized_test):,} docs")
else:
    t0 = time.time()
    print(f"Tokenizing {len(test_summaries):,} test docs...")
    with Pool(4) as p:
        tokenized_test = list(tqdm(
            p.imap(tokenize_one, test_summaries, chunksize=100),
            total=len(test_summaries), desc="Tokenize test",
        ))
    print(f"  Done: {time.time()-t0:.1f}s")
    with open(cache_test, "wb") as f:
        pickle.dump(tokenized_test, f)
    print(f"  Saved: {cache_test}")

tokenized_test_map = {item.summary: tok for item, tok in zip(test, tokenized_test)}

print(f"Sample original:  {documents[0][:100]}")
print(f"Sample tokenized: {tokenized_train[0][:100]}")

Loading cached: day3\tokenized_train_1m.pkl
  Loaded 85,727 docs
Loading cached: day3\tokenized_test_1m.pkl
  Loaded 3,872 docs
Sample original:  Tiêu đề: Áo len hoodie dày ấm cho nữ  
Danh mục: Thời trang nữ  
Thương hiệu: LiLiLa  
Mô tả: Hoodie
Sample tokenized: Tiêu_đề : Áo len_hoodie dày ấm cho nữ Danh_mục : Thời_trang nữ Thương_hiệu : LiLiLa Mô_tả : Hoodie c


### 5b. TF-IDF Architecture B + LR

In [7]:
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=10000)
X_train_b = vectorizer_b.fit_transform(tokenized_train)
print(f"TF-IDF B: {X_train_b.shape[0]:,} x {X_train_b.shape[1]:,} ({time.time()-t0:.1f}s)")

selected_words_b = vectorizer_b.get_feature_names_out()
print(f"Sample features: {list(selected_words_b[5000:5020])}")

TF-IDF B: 85,727 x 10,000 (2.5s)
Sample features: ['lạng', 'lạnh', 'lạnh_mát', 'lạnh_ren', 'lạp_xưởng', 'lấp_lánh', 'lấy', 'lần', 'lẩu', 'lẩu_thái', 'lẫn', 'lập', 'lập_trình', 'lật', 'lắc', 'lắk', 'lắp', 'lắp_ghép', 'lắp_ráp', 'lắp_đặt']


In [16]:
t0 = time.time()
lr_model_b = LinearRegression()
lr_model_b.fit(X_train_b, prices)
print(f"LR B train: {time.time()-t0:.1f}s")

def lr_tfidf_arch_b(item):
    tokenized = tokenized_test_map.get(item.summary) or tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(lr_model_b.predict(x)[0], 0)

results_lr_b = evaluate(lr_tfidf_arch_b, test)

LR B train: 2.8s


  0%|          | 0/200 [00:00<?, ?it/s]

13,742 10,376 116,486 30,670 48,770 225,893 160,504 72,918 313,744 228,329 21,356 83,052 119,299 213,821 51,381 66,523 131,786 122,548 101,456 16,733 51,726 63,750 107,935 46,714 171,833 93,448 173,812 78,957 482,533 44,772 143,096 62,465 58,629 1,868 121,761 103,037 29,230 33,370 370,392 466,748 96,413 4,295 129,129 83,718 144,839 472,634 50,000 37,107 86,000 92,099 47,261 51,118 95,242 417,463 127,859 24,915 102,747 14,205 115,553 263,805 265,546 48,536 67,857 117,109 173,724 85,619 17,674 31,974 163,867 2,053 80,154 75,127 34,381 48,785 45,329 9,640 130,900 19,584 46,097 150,001 232,755 508,395 204,455 183,019 209,096 158,293 87,644 93,050 6,936 39,595 31,825 57,640 152,517 436,023 58,988 33,949 78,869 147,353 181,361 314,035 218,377 179,000 57,515 54,202 120,753 67,966 33,762 7,417 40,546 54,335 100,567 87,054 272,548 360,364 94,610 30,757 7,311 518,401 77,681 6,364 38,926 150,133 66,791 85,076 97,764 83,615 54,004 169,142 171,190 17,093 135,829 163,056 168,356 29,025 33,121 44,637

---
## Step 6: Ensemble Models (dung Architecture B)

Tat ca ensemble dung  (underthesea tokenized) +  (da cache).

### 6a. Random Forest (subset 40K, 300 trees)

In [17]:
SUBSET_RF = 40_000           
t0 = time.time()                                                                                                                
rf_model = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=6, verbose=1)
rf_model.fit(X_train_b[:SUBSET_RF], prices[:SUBSET_RF])                                                                         
print(f"\nRF train: {time.time()-t0:.1f}s ({SUBSET_RF:,} samples)")                                                           
                                                                                                                                
def random_forest_pricer(item):                                                                                                 
    tokenized = tokenized_test_map.get(item.summary) or tokenize_one(item.summary)                                              
    x = vectorizer_b.transform([tokenized])                                                                                   
    return max(0, rf_model.predict(x)[0])
                                                                                                                                
results_rf = evaluate(random_forest_pricer, test)

[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:  3.9min
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed: 16.9min



RF train: 1637.1s (40,000 samples)


[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed: 27.3min finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.


  0%|          | 0/200 [00:00<?, ?it/s]

[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.1s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Using ba

...

[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed: 

560,218 166,923 12,267 189,928 14,785 44,456 

[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[

77,415 20,589 14,832 102,595 167,820 93,542 114,247 
Random Forest Pricer Results (200 items):
  RMSLE:  0.6360
  MAE:    128,385 VND
  MAPE:   66.9%
  R2:     41.1%


[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 300 out of 300 | elapsed:    0.0s finished


### 6b. XGBoost (full data, GPU)

In [8]:
t0 = time.time()                                                                                                                
xgb_model = xgb.XGBRegressor(                                                                                                   
    n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6,                                                          
    tree_method="hist",                                                                                                         
)                                                                                                                               
                                                                                                                                
pbar_xgb = tqdm(total=1000, desc="XGBoost")                                                                                     

class XGBProgress(xgb.callback.TrainingCallback):                                                                               
    def after_iteration(self, model, epoch, evals_log):   
        pbar_xgb.update(1)
        return False                                                                                                            
    def after_training(self, model):
        pbar_xgb.close()                                                                                                        
        return model                                      

xgb_model.set_params(callbacks=[XGBProgress()])                                                                                 
xgb_model.fit(X_train_b, prices)
print(f"XGBoost train: {time.time()-t0:.1f}s")                                                                                  
                                                        
def xgboost_pricer(item):
    tokenized = tokenized_test_map.get(item.summary) or tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])                                                                                     
    return max(0, xgb_model.predict(x)[0])
                                                                                                                                
results_xgb = evaluate(xgboost_pricer, test)                                                                                    

XGBoost:   0%|          | 0/1000 [00:00<?, ?it/s]

XGBoost train: 213.7s


  0%|          | 0/200 [00:00<?, ?it/s]

189,379 1,445 184,944 15,014 79,567 403,790 7,439 52,593 280,140 124,576 17,261 14,490 197,230 248,276 112,095 3,496 127,062 128,994 112,006 16,794 123,995 149,550 65,839 14,223 235,290 35,407 19,165 18,856 556,774 161,291 90,132 201,350 113,600 63,735 61,451 210,735 54,401 78,091 608,601 446,019 125,677 39,791 162,638 20,749 131,389 388,479 90,651 7,942 51,275 60,268 175,437 46,829 203,734 444,975 154,057 37,561 144,088 57,504 3,653 120,973 426,386 52,379 27,423 102,734 48,587 33,434 36,286 29,337 248,667 80,504 27,752 56,651 7,837 76,901 84,400 65,388 46,881 17,487 24,752 5,691 59,488 428,021 21,614 324,813 177,813 123,979 20,204 80,099 108,754 34,621 108,493 131,098 114,337 430,865 57,359 11,944 53,653 284,104 131,186 199,198 135,909 40,216 39,115 151,928 194,434 19,645 32,066 127,915 50,760 15,491 26,331 51,497 404,559 268,452 61,418 135,707 20,773 552,428 45,390 42,907 144,367 103,440 44,381 87,181 146,026 20,817 29,126 104,880 213,237 24,373 140,899 148,127 227,700 253 72,430 111

### 6c. LightGBM (full data)

In [9]:
t0 = time.time()
pbar_lgb = tqdm(total=1000, desc="LightGBM")

def lgb_tqdm_cb(env):
    pbar_lgb.update(1)

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6, verbose=-1,
)
lgb_model.fit(X_train_b, prices, callbacks=[lgb_tqdm_cb])
pbar_lgb.close()
print(f"LightGBM train: {time.time()-t0:.1f}s")

def lightgbm_pricer(item):
    tokenized = tokenized_test_map.get(item.summary) or tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(0, lgb_model.predict(x)[0])

results_lgb = evaluate(lightgbm_pricer, test)

LightGBM:   0%|          | 0/1000 [00:00<?, ?it/s]

LightGBM train: 41.4s


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  0%|          | 0/200 [00:00<?, ?it/s]

196,297 964 156,923 3,204 31,761 383,555 155,614 18,143 250,128 94,649 1,331 17,299 142,205 35,631 77,034 13,021 164,804 100,644 133,654 40,014 83,976 119,615 69,444 31,416 195,859 8,406 41,602 9,945 552,676 63,626 124,500 147,199 33,529 24,508 64,138 146,767 46,260 45,611 548,145 427,348 126,133 42,012 158,361 9,538 126,075 382,025 30,437 24,561 21,635 51,759 64,742 45,094 194,226 424,822 211,367 14,322 175,974 45,692 30,219 118,296 225,864 95,583 17,235 97,216 111,387 39,225 93,711 50,829 160,278 52,922 43,691 45,378 8,740 39,567 83,406 6,363 78,416 9,373 2,862 112,037 14,130 496,659 36,806 235,637 177,024 100,575 19,609 12,862 130,474 6,962 3,663 30,338 143,171 336,542 66,778 58,622 76,885 164,780 116,379 319,442 154,679 35,043 532 190,541 133,867 60,332 35,797 74,613 6,755 27,440 2,540 85,202 306,340 440,752 106 58,005 90,507 577,995 1,947 21,041 57,991 118,191 81,708 39,802 80,866 23,717 5,246 77,632 437,485 46,526 137,259 84,111 233,892 15,472 40,306 129,105 89,137 104,295 5,335 

### 6d. CatBoost (full data, GPU)

In [10]:
t0 = time.time()                                                                                                                
cb_model = CatBoostRegressor(                             
    iterations=1000, learning_rate=0.1, random_seed=SEED, verbose=100,
)
cb_model.fit(X_train_b, prices)                                                                                                 
print(f"CatBoost train: {time.time()-t0:.1f}s")

0:	learn: 225771.8822145	total: 337ms	remaining: 5m 36s
100:	learn: 197649.9864960	total: 16.3s	remaining: 2m 25s
200:	learn: 188554.0469659	total: 32s	remaining: 2m 7s
300:	learn: 182473.6319293	total: 47.6s	remaining: 1m 50s
400:	learn: 178015.3841138	total: 1m 3s	remaining: 1m 34s
500:	learn: 174387.4204154	total: 1m 18s	remaining: 1m 18s
600:	learn: 171185.5403691	total: 1m 34s	remaining: 1m 2s
700:	learn: 168385.1545012	total: 1m 49s	remaining: 46.9s
800:	learn: 165926.8082210	total: 2m 6s	remaining: 31.3s
900:	learn: 163559.0324949	total: 2m 21s	remaining: 15.6s
999:	learn: 161464.9583324	total: 2m 37s	remaining: 0us
CatBoost train: 158.1s


In [11]:
def cbboost_pricer(item):
    tokenized = tokenized_test_map.get(item.summary) or tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])                                                                                     
    return max(0, cb_model.predict(x)[0])
                                                                                                                                
results_cb = evaluate(cbboost_pricer, test)  

Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000
Object info sizes: 1 10000


  0%|          | 0/200 [00:00<?, ?it/s]

170,847 47,046 197,794 19,608 103,100 317,116 5,666 3,690 264,660 134,050 9,401 43,230 165,667 87,663 60,973 5,345 115,283 132,765 86,277 10,862 136,184 235,020 48,963 10,900 249,156 34,059 69,718 10,877 562,533 136,188 133,583 199,996 156,430 72,220 64,733 173,898 65,561 83,240 560,201 403,922 152,931 57,515 154,669 16,424 104,869 403,808 86,207 9,711 39,801 65,802 151,919 43,715 184,511 505,030 159,461 38,116 109,977 71,484 53,998 110,926 369,897 50,183 35,715 96,427 49,817 64 57,633 8,722 197,101 101,179 40,507 95,224 41,042 62,082 89,126 47,203 11,941 44,704 8,782 40,750 79,565 429,028 21,089 335,619 214,278 125,536 7,984 77,350 95,981 9,342 23,647 52,727 109,467 445,933 68,889 11,505 67,193 240,305 159,898 213,309 173,262 51,434 26,562 129,999 149,446 30,773 29,931 128,868 20,269 10,924 15,811 14,976 451,343 281,894 29,882 113,324 45,789 522,588 53,004 54,152 34,752 86,459 38,268 83,217 153,229 62,620 61,676 127,375 317,014 56,839 120,956 125,695 207,543 4,044 55,405 146,589 130,7

---
## Final Summary

In [ ]:
all_results = [
    ("Random", results_random),
    ("Mean", results_mean),
    ("Median", results_median),
    ("LR + TF-IDF (Arch A)", results_lr_a),
    ("LR + TF-IDF (Arch B)", results_lr_b),
    ("Random Forest (Arch B)", results_rf),
    ("XGBoost (Arch B)", results_xgb),
    ("LightGBM (Arch B)", results_lgb),
    ("CatBoost (Arch B)", results_cb),
]

summary = pd.DataFrame([
    {"Model": name, **res} for name, res in all_results
])
summary = summary.sort_values("rmsle")
print(f"FINAL SUMMARY -- Day 3 Baseline ML (<= {PRICE_THRESHOLD:,} VND)")
print("=" * 75)
print(summary.to_string(index=False))

best_name, best_res = min(all_results, key=lambda x: x[1]["rmsle"])
print(f"Best model (RMSLE): {best_name} -- RMSLE={best_res['rmsle']:.4f}")